In [ ]:
from collections import Counter
import heapq
import math
import numpy as np
from scipy.fft import dct, idct
import matplotlib.pyplot as plt
from struct import unpack, pack

: 

In [ ]:
def nuB(x: int):
    lims = np.arange(3)
    lims = 2**lims
    lims *= 8
    lims = 2**lims
    for nB, lim in enumerate(lims.tolist()):
        if x < lim:
            return 2**nB
    return 2**3


nuB(1e9)

In [ ]:
f = open("Wildlife.wmv", "rb")
lenchunk = 0x100##########
chunk = f.read(lenchunk)
if len(chunk) < lenchunk:
    chunk += b"\x00" * (lenchunk - len(chunk))
x = unpack("<" + "b" * lenchunk, chunk)
x = np.array(x)
x

In [ ]:
coef = dct(x, n=lenchunk)
abscoef = np.abs(coef)
scoef = np.sort(abscoef)
n_eliminated = lenchunk*1//2  ##########
absthreshold = scoef[n_eliminated]
fcoef = np.where(np.abs(coef) < absthreshold, 0, coef)
kcoef = min(0x7Fff / fcoef.max(), -0x8000 / fcoef.min())##########
ncoef = np.rint(fcoef * kcoef).astype(int)
lenchunk - n_eliminated, kcoef, unpack("d", pack("d", kcoef)), ncoef.max(), ncoef.min()

In [ ]:
reconstructed = idct(ncoef / kcoef, n=lenchunk)
reconstructed = np.rint(reconstructed).astype(np.int64)
reconstructed

In [ ]:
dxr = x - reconstructed
plt.figure(figsize=(12, 4))
plt.plot(dxr)
plt.xlabel("Sample index")
plt.ylabel("Error (x - reconstructed)")
plt.title("Reconstruction error")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
dxr_freq = Counter(int(v) for v in dxr)
if not dxr_freq:
    raise ValueError("dxr is empty")

huff_heap = []
huff_serial = 0

for symbol, frequency in dxr_freq.items():
    heapq.heappush(
        huff_heap,
        (frequency, huff_serial, {"symbol": symbol, "left": None, "right": None}),
    )
    huff_serial += 1

while len(huff_heap) > 1:
    freq_a, _, node_a = heapq.heappop(huff_heap)
    freq_b, _, node_b = heapq.heappop(huff_heap)
    parent = {"symbol": None, "left": node_a, "right": node_b}
    heapq.heappush(huff_heap, (freq_a + freq_b, huff_serial, parent))
    huff_serial += 1

huff_root = huff_heap[0][2]
huff_lengths = {}
huff_leaf_positions = []
huff_edges = []
huff_next_x = 0


def layout_huffman(node, depth=0):
    global huff_next_x

    if node["symbol"] is not None:
        x_pos = huff_next_x
        huff_next_x += 1
        huff_lengths[node["symbol"]] = max(1, depth)
        huff_leaf_positions.append((x_pos, -depth, node["symbol"]))
        return x_pos

    left_x = layout_huffman(node["left"], depth + 1)
    right_x = layout_huffman(node["right"], depth + 1)
    parent_x = (left_x + right_x) / 2
    huff_edges.extend(
        [
            ((parent_x, -depth), (left_x, -(depth + 1))),
            ((parent_x, -depth), (right_x, -(depth + 1))),
        ]
    )
    return parent_x


layout_huffman(huff_root)

huff_bits = sum(
    frequency * huff_lengths[symbol] for symbol, frequency in dxr_freq.items()
)
huff_bytes = (huff_bits + 7) // 8
print(f"Symbols: {len(dxr_freq)}")
print(f"Encoded payload: {huff_bits:,} bits = {huff_bytes:,} bytes")

fig, ax = plt.subplots(figsize=(max(12, len(dxr_freq) * 0.18), 8))
for (x1, y1), (x2, y2) in huff_edges:
    ax.plot([x1, x2], [y1, y2], color="gray", linewidth=0.7)

for x_pos, y_pos, symbol in huff_leaf_positions:
    ax.text(x_pos, y_pos - 0.15, str(symbol), ha="center", va="top", fontsize=7)

ax.set_title("Huffman tree for dxr (leaf labels are symbols)")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
adxr = np.abs(dxr)
adxr_freq = Counter(int(v) for v in adxr)

adxr_heap = []
adxr_serial = 0
for symbol, frequency in adxr_freq.items():
    heapq.heappush(
        adxr_heap,
        (frequency, adxr_serial, {"symbol": symbol, "left": None, "right": None}),
    )
    adxr_serial += 1

while len(adxr_heap) > 1:
    freq_a, _, node_a = heapq.heappop(adxr_heap)
    freq_b, _, node_b = heapq.heappop(adxr_heap)
    parent = {"symbol": None, "left": node_a, "right": node_b}
    heapq.heappush(adxr_heap, (freq_a + freq_b, adxr_serial, parent))
    adxr_serial += 1

adxr_root = adxr_heap[0][2]
adxr_lengths = {}
adxr_edges = []
adxr_leaves = []
adxr_next_x = 0


def layout_adxr(node, depth=0):
    global adxr_next_x

    if node["symbol"] is not None:
        x_pos = adxr_next_x
        adxr_next_x += 1
        adxr_lengths[node["symbol"]] = max(1, depth)
        adxr_leaves.append((x_pos, -depth, node["symbol"]))
        return x_pos

    left_x = layout_adxr(node["left"], depth + 1)
    right_x = layout_adxr(node["right"], depth + 1)
    parent_x = (left_x + right_x) / 2
    adxr_edges.extend(
        [
            ((parent_x, -depth), (left_x, -(depth + 1))),
            ((parent_x, -depth), (right_x, -(depth + 1))),
        ]
    )
    return parent_x


layout_adxr(adxr_root)

adxr_bits = sum(
    frequency * adxr_lengths[symbol] for symbol, frequency in adxr_freq.items()
)
adxr_payload_bytes = (adxr_bits + 7) // 8
adxr_base_bytes = lenchunk // 8
adxr_total_bytes = adxr_base_bytes + adxr_payload_bytes

print(f"Symbols: {len(adxr_freq)}")
print(f"Huffman payload: {adxr_bits:,} bits = {adxr_payload_bytes:,} bytes")
print(f"Base storage: {adxr_base_bytes:,} bytes")
print(f"Total: {adxr_total_bytes:,} bytes (excluding Huffman-tree metadata)")

fig, ax = plt.subplots(figsize=(max(12, len(adxr_freq) * 0.18), 8))
for (x1, y1), (x2, y2) in adxr_edges:
    ax.plot([x1, x2], [y1, y2], color="gray", linewidth=0.7)

for x_pos, y_pos, symbol in adxr_leaves:
    ax.text(x_pos, y_pos - 0.15, str(symbol), ha="center", va="top", fontsize=7)

ax.set_title("Huffman tree for adxr (leaf labels are symbols)")
ax.axis("off")
plt.tight_layout()
plt.show()